In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import ipywidgets as widgets
from IPython.display import display

class HodgkinHuxleyInteractive:
    """Hodgkin-Huxley Model with adjustable parameters for interaction."""
    C_m = 1.0
    t = np.arange(0.0, 450.0, 0.01)

    # Kinetics functions from your baseline model
    def alpha_m(self, V): return 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))
    def beta_m(self, V): return 4.0 * np.exp(-(V + 65.0) / 18.0)
    def alpha_h(self, V): return 0.07 * np.exp(-(V + 65.0) / 20.0)
    def beta_h(self, V): return 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))
    def alpha_n(self, V): return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))
    def beta_n(self, V): return 0.125 * np.exp(-(V + 65) / 80.0)

    def I_inj(self, t): 
        return 10 * (t > 100) - 10 * (t > 200) + 35 * (t > 300) - 35 * (t > 400)

    def dALLdt(self, X, t, g_Na, g_K, g_L, E_Na, E_K, E_L):
        V, m, h, n = X
        dVdt = (self.I_inj(t) - (g_Na*m**3*h*(V-E_Na)) - (g_K*n**4*(V-E_K)) - (g_L*(V-E_L))) / self.C_m
        dmdt = self.alpha_m(V)*(1.0-m) - self.beta_m(V)*m
        dhdt = self.alpha_h(V)*(1.0-h) - self.beta_h(V)*h
        dndt = self.alpha_n(V)*(1.0-n) - self.beta_n(V)*n
        return [dVdt, dmdt, dhdt, dndt]

def plot_hh(g_Na, g_K, g_L, E_Na, E_K, E_L):
    hh = HodgkinHuxleyInteractive()
    # Initial conditions
    X = odeint(hh.dALLdt, [-65, 0.05, 0.6, 0.32], hh.t, args=(g_Na, g_K, g_L, E_Na, E_K, E_L))
    V, m, h, n = X[:,0], X[:,1], X[:,2], X[:,3]
    
    # Set plot style to Pink
    fig, axs = plt.subplots(4, 1, figsize=(10, 12), facecolor='#FFC0CB')
    for ax in axs: ax.set_facecolor('#FFF0F5')

    # 1. Voltage
    axs[0].plot(hh.t, V, 'k')
    axs[0].set_ylabel('V (mV)')
    axs[0].set_title('Hodgkin-Huxley: Pink Interactive Simulation')

    # 2. Ionic Currents
    axs[1].plot(hh.t, g_Na*m**3*h*(V-E_Na), 'c', label='$I_{Na}$')
    axs[1].plot(hh.t, g_K*n**4*(V-E_K), 'y', label='$I_{K}$')
    axs[1].plot(hh.t, g_L*(V-E_L), 'm', label='$I_{L}$')
    axs[1].set_ylabel('Current ($\mu A/cm^2$)')
    axs[1].legend(loc='upper right')

    # 3. Gating Variables
    axs[2].plot(hh.t, m, 'r', label='m')
    axs[2].plot(hh.t, h, 'g', label='h')
    axs[2].plot(hh.t, n, 'b', label='n')
    axs[2].set_ylabel('Gating')
    axs[2].legend(loc='upper right')

    # 4. Injected Current
    axs[3].plot(hh.t, [hh.I_inj(t) for t in hh.t], 'k')
    axs[3].set_xlabel('Time (ms)')
    axs[3].set_ylabel('$I_{inj}$')
    
    plt.tight_layout()
    plt.show()

# Interactive Dashboard
widgets.interact(
    plot_hh,
    g_Na=widgets.FloatSlider(value=120, min=50, max=200, step=5, description='g_Na'),
    g_K=widgets.FloatSlider(value=36, min=10, max=100, step=2, description='g_K'),
    g_L=widgets.FloatSlider(value=0.3, min=0.1, max=1.0, step=0.1, description='g_L'),
    E_Na=widgets.FloatSlider(value=50, min=30, max=80, description='E_Na'),
    E_K=widgets.FloatSlider(value=-77, min=-100, max=-50, description='E_K'),
    E_L=widgets.FloatSlider(value=-54.4, min=-70, max=-40, description='E_L')
);

interactive(children=(FloatSlider(value=120.0, description='g_Na', max=200.0, min=50.0, step=5.0), FloatSlider…